In [1]:
"""
Adım 3: Spark Structured Streaming + Delta Lake
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("ClimateStreaming")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.streaming.checkpointLocation", "/home/jovyan/work/delta-lake/_checkpoints")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark sürümü: {spark.version}")
print("Spark Session hazır ✅")

Spark sürümü: 3.5.0
Spark Session hazır ✅


In [2]:
# Kafka'dan gelecek JSON mesajının şeması
# Producer'dakiyle bire bir eşleşmeli — bu şema sözleşmesi (data contract)
climate_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("city_name", StringType(), True),
    StructField("measurement_date", StringType(), True),
    StructField("season", StringType(), True),
    StructField("avg_temp_c", DoubleType(), True),
    StructField("min_temp_c", DoubleType(), True),
    StructField("max_temp_c", DoubleType(), True),
    StructField("precipitation_mm", DoubleType(), True),
    StructField("snow_depth_mm", DoubleType(), True),
    StructField("avg_wind_dir_deg", DoubleType(), True),
    StructField("avg_wind_speed_kmh", DoubleType(), True),
    StructField("peak_wind_gust_kmh", DoubleType(), True),
    StructField("avg_sea_level_pres_hpa", DoubleType(), True),
    StructField("sunshine_total_min", DoubleType(), True),
])

KAFKA_BOOTSTRAP = "kafka:9092"
KAFKA_TOPIC = "climate-events"

print("Şema tanımlandı ✅")

Şema tanımlandı ✅


In [3]:
# Önce statik (batch) okuma ile test edelim
# Streaming'e geçmeden Kafka bağlantısının çalıştığını doğrularız
test_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print(f"Kafka'dan okunan mesaj sayısı: {test_df.count()}")
test_df.printSchema()

Kafka'dan okunan mesaj sayısı: 5000
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
# Kafka'dan gelen "value" alanı binary, içinde JSON var
# String'e çevirip from_json ile şemamıza göre parse edelim
parsed_test = (
    test_df
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_timestamp", "offset")
    .select(
        from_json(col("json_str"), climate_schema).alias("data"),
        col("kafka_timestamp"),
        col("offset")
    )
    .select("data.*", "kafka_timestamp", "offset")
)

print(f"Parse edilmiş satır sayısı: {parsed_test.count()}")
parsed_test.printSchema()
parsed_test.show(5, truncate=False)

Parse edilmiş satır sayısı: 5000
root
 |-- timestamp: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- measurement_date: string (nullable = true)
 |-- season: string (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- snow_depth_mm: double (nullable = true)
 |-- avg_wind_dir_deg: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)
 |-- peak_wind_gust_kmh: double (nullable = true)
 |-- avg_sea_level_pres_hpa: double (nullable = true)
 |-- sunshine_total_min: double (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- offset: long (nullable = true)

+--------------------------------+-------------------+----------+---------+-------------------+------+----------+----------+----------+----------------+

In [5]:
# STREAMING okuma — readStream kullanıyoruz
# Kafka'yı sürekli dinler, yeni mesaj geldikçe işler
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")  # baştan oku (test için)
    .option("failOnDataLoss", "false")      # offset kaybolursa hata verme
    .load()
)

# Bronze için hazırlık: Kafka metadata'sını koru, JSON'u parse et
bronze_stream = (
    stream_df
    .selectExpr(
        "CAST(value AS STRING) as json_str",
        "topic",
        "partition",
        "offset",
        "timestamp as kafka_timestamp"
    )
    .select(
        from_json(col("json_str"), climate_schema).alias("data"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("kafka_timestamp"),
        col("json_str").alias("raw_json")  # ham JSON'u sakla (debug ve audit için)
    )
    .select(
        "data.*",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        "raw_json",
        current_timestamp().alias("ingestion_time")  # ne zaman Spark'a girdi
    )
)

print("Streaming kaynağı hazır ✅")
print("Streaming mi? →", bronze_stream.isStreaming)
bronze_stream.printSchema()

Streaming kaynağı hazır ✅
Streaming mi? → True
root
 |-- timestamp: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- measurement_date: string (nullable = true)
 |-- season: string (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- snow_depth_mm: double (nullable = true)
 |-- avg_wind_dir_deg: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)
 |-- peak_wind_gust_kmh: double (nullable = true)
 |-- avg_sea_level_pres_hpa: double (nullable = true)
 |-- sunshine_total_min: double (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- raw_json: string (nullable = true)
 |-- ingestion_time:

In [6]:
# BRONZE: Ham veriyi Delta Lake'e yaz
# Delta = Parquet + transaction log (ACID, time travel, schema enforcement)

BRONZE_PATH = "/home/jovyan/work/delta-lake/bronze/climate_events"
BRONZE_CHECKPOINT = "/home/jovyan/work/delta-lake/_checkpoints/bronze"

bronze_query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")  # Bronze'a sadece yeni veri ekle, güncelleme yok
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")  # her 10 saniyede bir batch işle
    .start(BRONZE_PATH)
)

print(f"Bronze stream başladı ✅")
print(f"  Path       : {BRONZE_PATH}")
print(f"  Checkpoint : {BRONZE_CHECKPOINT}")
print(f"  Query ID   : {bronze_query.id}")
print(f"  Aktif mi?  : {bronze_query.isActive}")

Bronze stream başladı ✅
  Path       : /home/jovyan/work/delta-lake/bronze/climate_events
  Checkpoint : /home/jovyan/work/delta-lake/_checkpoints/bronze
  Query ID   : 9b8251a0-8e09-47b0-8121-2c281e269b6f
  Aktif mi?  : True


In [7]:
# Bronze'a yazılanı oku ve doğrula
import time
time.sleep(15)  # Stream'in en az bir batch işlemesi için bekle

bronze_check = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze tablosundaki kayıt sayısı: {bronze_check.count()}")
bronze_check.show(3, truncate=False)

Bronze tablosundaki kayıt sayısı: 5000
+--------------------------------+-------------------+----------+---------+-------------------+------+----------+----------+----------+----------------+-------------+----------------+------------------+------------------+----------------------+------------------+--------------+---------+------+-----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+
|timestamp                       |event_type         |station_id|city_name|measurement_date   |season|avg_temp_c|min_temp_c|max_temp_c|precipitation_mm|snow_depth_mm|avg

## Silver Streaming

In [8]:
from pyspark.sql.functions import (
    col, to_date, to_timestamp, year, month, dayofmonth,
    when, lit, regexp_extract, length
)

# Bronze'dan streaming olarak oku
# (Yeni gelen veriler için sürekli işlemeye devam edecek)
bronze_to_silver = (
    spark.readStream
    .format("delta")
    .load(BRONZE_PATH)
)

# ----- Veri temizleme ve dönüştürmeler -----
silver_stream = (
    bronze_to_silver
    
    # 1) Tarih dönüşümü: string -> date
    .withColumn("measurement_date", to_date(col("measurement_date")))
    
    # 2) Timestamp dönüşümü: string -> timestamp (gönderim zamanı)
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    
    # 3) BUG FIX: season kolonu producer'da yanlışlıkla 'date' alıyor
    # Tarih formatı (YYYY-MM-DD) varsa season geçersiz, NULL yap
    .withColumn(
        "season",
        when(
            length(col("season")) > 10,  # gerçek season "Summer" gibi kısa olur
            lit(None)
        ).otherwise(col("season"))
    )
    
    # 4) Veri kalite filtreleri (iş kuralları)
    # Sıcaklık fiziksel sınırlar dışındaysa elenir
    .filter(
        (col("avg_temp_c").isNull()) |
        ((col("avg_temp_c") >= -90) & (col("avg_temp_c") <= 60))
    )
    .filter(
        (col("min_temp_c").isNull()) |
        ((col("min_temp_c") >= -90) & (col("min_temp_c") <= 60))
    )
    .filter(
        (col("max_temp_c").isNull()) |
        ((col("max_temp_c") >= -90) & (col("max_temp_c") <= 60))
    )
    
    # Tarih ve şehir olmayan kayıt anlamsız → at
    .filter(col("measurement_date").isNotNull())
    .filter(col("city_name").isNotNull())
    .filter(col("station_id").isNotNull())
    
    # 5) Zenginleştirme: tarih bileşenlerini ayır (analiz için faydalı)
    .withColumn("year",  year(col("measurement_date")))
    .withColumn("month", month(col("measurement_date")))
    .withColumn("day",   dayofmonth(col("measurement_date")))
    
    # 6) Audit metadata
    .withColumn("processed_at", current_timestamp())
    
    # 7) Silver'da artık raw_json ve Kafka offset gibi şeyler gerekmiyor — düşür
    .drop("raw_json", "topic", "partition", "offset", "timestamp")
)

print("Silver dönüşümleri tanımlandı ✅")
silver_stream.printSchema()

Silver dönüşümleri tanımlandı ✅
root
 |-- event_type: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- measurement_date: date (nullable = true)
 |-- season: string (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- snow_depth_mm: double (nullable = true)
 |-- avg_wind_dir_deg: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)
 |-- peak_wind_gust_kmh: double (nullable = true)
 |-- avg_sea_level_pres_hpa: double (nullable = true)
 |-- sunshine_total_min: double (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- processed_at: time

In [9]:
SILVER_PATH = "/home/jovyan/work/delta-lake/silver/climate_events"
SILVER_CHECKPOINT = "/home/jovyan/work/delta-lake/_checkpoints/silver"

silver_query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .option("mergeSchema", "true")
    # Yıl ve aya göre partition'la — sorgu performansı için önemli
    .partitionBy("year", "month")
    .trigger(processingTime="10 seconds")
    .start(SILVER_PATH)
)

print(f"Silver stream başladı ✅")
print(f"  Path       : {SILVER_PATH}")
print(f"  Aktif mi?  : {silver_query.isActive}")

Silver stream başladı ✅
  Path       : /home/jovyan/work/delta-lake/silver/climate_events
  Aktif mi?  : True


In [11]:
import time
time.sleep(15)

silver_check = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver tablosundaki kayıt sayısı: {silver_check.count()}")

# Veri kalitesi raporu
print("\n--- Veri Kalitesi ---")
print(f"Toplam kayıt:         {silver_check.count():,}")
print(f"Benzersiz şehir:      {silver_check.select('city_name').distinct().count()}")
print(f"Benzersiz istasyon:   {silver_check.select('station_id').distinct().count()}")
print(f"En eski tarih:        {silver_check.agg({'measurement_date': 'min'}).collect()[0][0]}")
print(f"En yeni tarih:        {silver_check.agg({'measurement_date': 'max'}).collect()[0][0]}")

# avg_temp_c için temel istatistikler
print("\n--- Sıcaklık İstatistikleri ---")
silver_check.select("avg_temp_c", "min_temp_c", "max_temp_c").describe().show()

silver_check.show(3, truncate=False)

Silver tablosundaki kayıt sayısı: 5000

--- Veri Kalitesi ---
Toplam kayıt:         5,000
Benzersiz şehir:      1
Benzersiz istasyon:   1
En eski tarih:        1957-07-01
En yeni tarih:        2010-11-29

--- Sıcaklık İstatistikleri ---
+-------+------------------+-----------------+------------------+
|summary|        avg_temp_c|       min_temp_c|        max_temp_c|
+-------+------------------+-----------------+------------------+
|  count|              5000|             3719|              4072|
|   mean|18.086119999999994|11.21486958859908|25.258423379174854|
| stddev| 9.243158688571155|8.351186129315645|10.408331544072867|
|    min|              -2.1|            -11.0|               1.3|
|    max|              37.8|             28.9|              48.0|
+-------+------------------+-----------------+------------------+

+-------------------+----------+---------+----------------+------+----------+----------+----------+----------------+-------------+----------------+------------------+--

## Gold

In [12]:
from pyspark.sql.functions import (
    avg, sum as spark_sum, count, max as spark_max, min as spark_min,
    expr, dayofweek, weekofyear, quarter,
    when, col, lit
)
from pyspark.sql.window import Window
import pyspark.sql.functions as F

print("Gold imports hazır ✅")

Gold imports hazır ✅


In [13]:
# GOLD #1: ML-ready feature tablosu
# Silver'daki her satıra anlamlı feature'lar ekleyerek ML modelinin
# hazır kullanabileceği hale getiriyoruz.

silver_to_gold = (
    spark.readStream
    .format("delta")
    .load(SILVER_PATH)
)

gold_features_stream = (
    silver_to_gold
    
    # ===== Zaman bazlı feature'lar =====
    .withColumn("day_of_week", dayofweek(col("measurement_date")))   # 1=Pazar, 7=Cumartesi
    .withColumn("week_of_year", weekofyear(col("measurement_date"))) # 1-52
    .withColumn("quarter", quarter(col("measurement_date")))         # 1-4
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), 1).otherwise(0))
    
    # ===== Sıcaklık türevleri =====
    # Günün sıcaklık aralığı — ısı stresi göstergesi
    .withColumn("temp_range_c", col("max_temp_c") - col("min_temp_c"))
    
    # Sıcaklık ortalaması (eğer avg yoksa min-max ortalaması ile doldur)
    .withColumn(
        "temp_avg_filled",
        when(col("avg_temp_c").isNotNull(), col("avg_temp_c"))
        .otherwise((col("max_temp_c") + col("min_temp_c")) / 2)
    )
    
    # Donma günü flag'i (avg <= 0)
    .withColumn(
        "is_freezing",
        when(col("temp_avg_filled") <= 0, 1).otherwise(0)
    )
    
    # Sıcak gün flag'i (max >= 30)
    .withColumn(
        "is_hot_day",
        when(col("max_temp_c") >= 30, 1).otherwise(0)
    )
    
    # ===== Yağış kategorisi =====
    .withColumn(
        "precipitation_category",
        when(col("precipitation_mm").isNull() | (col("precipitation_mm") == 0), "none")
        .when(col("precipitation_mm") < 2.5, "light")
        .when(col("precipitation_mm") < 10, "moderate")
        .when(col("precipitation_mm") < 50, "heavy")
        .otherwise("extreme")
    )
    
    # Yağışlı gün flag'i
    .withColumn(
        "is_rainy",
        when(col("precipitation_mm") > 0.1, 1).otherwise(0)
    )
    
    # ===== Rüzgar feature'ları =====
    .withColumn(
        "wind_category",
        when(col("avg_wind_speed_kmh").isNull(), "unknown")
        .when(col("avg_wind_speed_kmh") < 10, "calm")
        .when(col("avg_wind_speed_kmh") < 30, "moderate")
        .when(col("avg_wind_speed_kmh") < 60, "strong")
        .otherwise("storm")
    )
    
    # ===== Veri kalitesi feature'ı =====
    # Bir satırda kaç ölçüm null değil — modelin "veri kalitesi" sinyali
    .withColumn(
        "feature_completeness",
        (when(col("avg_temp_c").isNotNull(), 1).otherwise(0) +
         when(col("min_temp_c").isNotNull(), 1).otherwise(0) +
         when(col("max_temp_c").isNotNull(), 1).otherwise(0) +
         when(col("precipitation_mm").isNotNull(), 1).otherwise(0) +
         when(col("avg_wind_speed_kmh").isNotNull(), 1).otherwise(0) +
         when(col("avg_sea_level_pres_hpa").isNotNull(), 1).otherwise(0))
    )
)

print("Gold features dönüşümleri tanımlandı ✅")
gold_features_stream.printSchema()

Gold features dönüşümleri tanımlandı ✅
root
 |-- event_type: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- measurement_date: date (nullable = true)
 |-- season: string (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- snow_depth_mm: double (nullable = true)
 |-- avg_wind_dir_deg: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)
 |-- peak_wind_gust_kmh: double (nullable = true)
 |-- avg_sea_level_pres_hpa: double (nullable = true)
 |-- sunshine_total_min: double (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- processed_a

In [14]:
GOLD_FEATURES_PATH = "/home/jovyan/work/delta-lake/gold/climate_features"
GOLD_FEATURES_CHECKPOINT = "/home/jovyan/work/delta-lake/_checkpoints/gold_features"

gold_features_query = (
    gold_features_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", GOLD_FEATURES_CHECKPOINT)
    .option("mergeSchema", "true")
    .partitionBy("year", "month")
    .trigger(processingTime="10 seconds")
    .start(GOLD_FEATURES_PATH)
)

print(f"Gold features stream başladı ✅")
print(f"  Aktif mi? {gold_features_query.isActive}")

Gold features stream başladı ✅
  Aktif mi? True


In [15]:
import time
time.sleep(20)

gold_features_check = spark.read.format("delta").load(GOLD_FEATURES_PATH)
print(f"Gold features kayıt sayısı: {gold_features_check.count()}")

print("\n--- Yeni Üretilen Feature'lar ---")
gold_features_check.select(
    "city_name", "measurement_date",
    "temp_avg_filled", "temp_range_c",
    "is_freezing", "is_hot_day", "is_rainy", "is_weekend",
    "precipitation_category", "wind_category",
    "feature_completeness"
).show(5, truncate=False)

print("\n--- Feature Dağılımları ---")
print("Yağış kategorisi dağılımı:")
gold_features_check.groupBy("precipitation_category").count().show()

print("Rüzgar kategorisi dağılımı:")
gold_features_check.groupBy("wind_category").count().show()

print("Veri tamlığı dağılımı (kaç sütun dolu):")
gold_features_check.groupBy("feature_completeness").count().orderBy("feature_completeness").show()

AnalysisException: [DELTA_TABLE_NOT_FOUND] Delta table `/home/jovyan/work/delta-lake/gold/climate_features` doesn't exist.

## Gold 2

In [ ]:
from pyspark.sql.functions import (
    avg, sum as _sum, count, max as _max, min as _min, 
    stddev, expr, round as _round
)

# Silver'dan static (batch) okuma — agregasyon için daha temiz yol
# Streaming agregasyon mümkün ama dashboard için tek seferlik özet yeterli
silver_static = spark.read.format("delta").load(SILVER_PATH)

city_monthly_summary = (
    silver_static
    .groupBy("city_name", "year", "month")
    .agg(
        # Sıcaklık
        _round(avg("avg_temp_c"), 2).alias("avg_temp_c"),
        _round(_min("min_temp_c"), 2).alias("min_temp_c"),
        _round(_max("max_temp_c"), 2).alias("max_temp_c"),
        _round(stddev("avg_temp_c"), 2).alias("temp_stddev"),
        
        # Yağış
        _round(_sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        _round(avg("precipitation_mm"), 2).alias("avg_precipitation_mm"),
        count(when(col("precipitation_mm") > 0.1, 1)).alias("rainy_days"),
        
        # Rüzgar
        _round(avg("avg_wind_speed_kmh"), 2).alias("avg_wind_speed"),
        _round(_max("peak_wind_gust_kmh"), 2).alias("peak_gust_kmh"),
        
        # Veri kalitesi
        count("*").alias("days_recorded"),
        count("avg_temp_c").alias("days_with_temp"),
    )
    # Veri tamlığı oranı (yüzde)
    .withColumn(
        "data_completeness_pct",
        _round((col("days_with_temp") / col("days_recorded")) * 100, 1)
    )
    .orderBy("city_name", "year", "month")
)

GOLD_SUMMARY_PATH = "/home/jovyan/work/delta-lake/gold/city_monthly_summary"

# Static yazma — overwrite mode (her çalıştırmada yeniden hesapla)
(
    city_monthly_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SUMMARY_PATH)
)

print(f"Gold city_monthly_summary yazıldı ✅")
print(f"  Path: {GOLD_SUMMARY_PATH}")
print(f"  Satır sayısı: {city_monthly_summary.count()}")

In [ ]:
summary_check = spark.read.format("delta").load(GOLD_SUMMARY_PATH)

print(f"Şehir-ay özeti satır sayısı: {summary_check.count()}")
print(f"\nKapsanan dönem:")
print(f"  En eski: {summary_check.agg(_min('year'), _min('month')).collect()[0]}")
print(f"  En yeni: {summary_check.agg(_max('year'), _max('month')).collect()[0]}")

print("\n--- Örnek satırlar (en sıcak aylar) ---")
summary_check.orderBy(col("avg_temp_c").desc()).show(5)

print("\n--- Örnek satırlar (en yağışlı aylar) ---")
summary_check.orderBy(col("total_precipitation_mm").desc()).show(5)

In [ ]:
'''# Aktif stream'leri durdur
for q in spark.streams.active:
    print(f"Durduruluyor: {q.id}")
    q.stop()

# Spark Session'u da kapat (container restart sonrası yeniden açacağız)
spark.stop()
print("Tüm stream'ler durduruldu, Spark Session kapatıldı ✅")'''